In [1]:
%env CUDA_VISIBLE_DEVICES=1

import logging
from typing import *

import hkkang_utils.file as file_utils
import torch

from colbert.noun_extraction.utils import unidecode_text
from colbert.utils.utils import isint
from model.late_encoder import ColBERTRetriever, RetrievalResult
from model.utils import Document
from scripts.evaluate.utils import get_recall_rates, load_beir_data

# Retriever path
ROOT = "/root/ColBERT/experiments/"
DATASET_DIR = "/root/ColBERT/data"
CHECKPOINT_DIR = "/root/ColBERT/checkpoint"
NBITS=2

DATASET_NAME = "hotpotqa"
MODEL_NAME = "baseline_nway64_q4"
SKIP_PADDING = True
IS_PHRASE = True

logger = logging.getLogger("TokenScore")

env: CUDA_VISIBLE_DEVICES=1


/home/user/.local/lib/python3.11/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/home/user/.local/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(


In [2]:
def load_collection(dataset_name: str) -> Dict:
    collection_path = f"{DATASET_DIR}/{dataset_name}/collection.tsv"
    return file_utils.read_csv_file(collection_path, delimiter="\t", first_row_as_header=True, quotechar=None)

In [3]:
def get_ranks(scores: List[float]) -> List[int]:
    ranks = []
    for i in range(len(scores)):
        rank = 1
        for j in range(len(scores)):
            if scores[j] > scores[i]:
                rank += 1
        ranks.append(rank)
    return ranks

In [4]:
def examine_token_scores(q_toks: List[str], d_toks: List[str], tok_scores: List[List[float]]) -> None:
    tok_scores = torch.tensor(tok_scores)
    # Find the max score
    max_scores = tok_scores.max(dim=1)[0]
    # Show max scores
    for q_i in range(len(q_toks)):
        if max_scores[q_i] != 0.0:
            print(f"Query token {q_i}: {q_toks[q_i]} (max score: {max_scores[q_i]})")
    print("\n")
    for q_i in range(len(q_toks)):
        if max_scores[q_i] != 0.0:
            print(f"Query token {q_i}: {q_toks[q_i]} (max score: {max_scores[q_i]})")
            # Rank the document tokens by score
            ranks = get_ranks(tok_scores[q_i])
            for d_i in range(len(d_toks)):
                if len(tok_scores[q_i]) <= d_i or len(d_toks) <= d_i:
                    continue
                print(f"\tDocument token {d_i}: {d_toks[d_i]} (score: {tok_scores[q_i][d_i]}, rank: {ranks[d_i]})")
            print("\n")
    return None

In [5]:
def show_passages(query: str, docs: List[Document], scores:List[float], gold_pids: List[int], gold_ranks:List[int], collection:Dict) -> None:
    assert len(docs) == len(scores), f"Length mismatch: {len(docs)} != {len(scores)}"
    print("Query: ", query)
    # Print gold documents
    for i, gold_pid in enumerate(gold_pids):
        gold_score = "unknow" if gold_ranks[i] == -1 else str(scores[gold_ranks[i]])
        print(f"Gold document {i}: (PID: {gold_pid}) (score: {gold_score})")
        print(f"{unidecode_text(collection[gold_pid][0])}\n")
    for i in range(10):
        doc = docs[i]
        print(f"Document {i+1}: {doc.title} (PID: {doc.id})  (score: {scores[i]})")
        print(f"{doc.text}\n")

In [6]:
def examine(retriever, query: str, pid: int, collection: Dict, is_phrase:bool=False) -> None:
    # Tokenizer for query texts
    q_tokenizer = retriever.searcher.checkpoint.query_tokenizer
    # Tokenizer for documents
    d_tokenizer = retriever.searcher.checkpoint.doc_tokenizer
    
    doc_text = unidecode_text(collection[pid][0])
    doc_title = unidecode_text(collection[pid][1])
    doc = Document(id=pid, text=doc_text, title=doc_title)
    if doc_title:
        doc_text = f"{doc_title} | {doc_text}"

    # Tokenize
    q_toks = q_tokenizer.tokenize([query], add_special_tokens=True)[0]
    d_toks = d_tokenizer.tokenize([doc_text], add_special_tokens=True)[0]
    
    # Convert tokens to phrases
    if is_phrase:
        # Get phrase indices
        q_phrase_indices = retriever.searcher.checkpoint.get_phrase_indices(texts=[query], tok_ids=None, masks=None, full_length_search=False, is_query=True)[0][0]
        d_phrase_indices = retriever.searcher.checkpoint.get_phrase_indices(texts=[doc_text], tok_ids=None, masks=None, full_length_search=False, is_query=False)[0][0]
        # Combine tokens into phrases
        q_toks = [" ".join(q_toks[s:e]) for s, e in q_phrase_indices]
        d_toks = [" ".join(d_toks[s:e]) for s, e in d_phrase_indices]
    
    # # Remove doc toks in skiplists
    # skiplist = list(key for key in retriever.searcher.checkpoint.skiplist.keys() if not isint(key))
    # d_toks = [d_tok for d_tok in d_toks if d_tok not in skiplist]    

    # Compute token scores
    result: RetrievalResult = retriever.calculate_score_by_doc_batch(queries=[query], docs=[doc])[0]

    print(f"Query: {query}")
    print(f"Document (score: {result.score}):\n{doc_text}\n")
    examine_token_scores(q_toks, d_toks, result.token_scores)

In [7]:
# Read in collection
collection: List[Dict[str, Any]] = load_collection(dataset_name=DATASET_NAME)
collection = {int(item['id']): [item["text"], item['title']] for item in collection}

In [10]:
# initialize retriever
index = f"{DATASET_NAME}.{MODEL_NAME}.nbits={NBITS}"
experiment = f"{DATASET_NAME}_unidecode"
retriever = ColBERTRetriever(root=ROOT, index=index, experiment=experiment, 
                                use_cache=IS_PHRASE and True,
                                skip_padding=SKIP_PADDING,
                                is_use_phrase_level=IS_PHRASE)

[Mar 06, 20:17:06] #> Loading collection...
0M 1M 2M 3M 4M 5M [Mar 06, 20:17:27] #> Loading codec...
[Mar 06, 20:17:28] Loading decompress_residuals_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
[Mar 06, 20:17:28] Loading packbits_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].


[Mar 06, 20:17:28] #> Loading IVF...
[Mar 06, 20:17:28] #> Loading doclens...


100%|██████████| 210/210 [00:00<00:00, 882.63it/s]


[Mar 06, 20:17:29] #> Loading codes and residuals...


100%|██████████| 210/210 [00:08<00:00, 24.93it/s]


In [11]:
# Load data
data = load_beir_data(dataset_dir=DATASET_DIR, dataset_name=DATASET_NAME, return_unique=True)
# Data as dictionary
data_dict = {}
for qid, query, pids, p_titles, scores in data:
    data_dict[qid] = (query, pids, p_titles, scores)
data = data_dict

In [41]:
target_qid = "5ae22b8d554299234fd0440f"
target_datum = data[target_qid]
query = target_datum[0]
gold_pids = [int(d) for d in target_datum[1]]
print(target_datum)

('What was the father of Kasper Schmeichel voted to be by the IFFHS in 1992?', ['590826', '165560'], [None, None], [1, 1])


In [42]:
# Get the query
# Retrieve top-100 passages for each query
all_results: List[List[RetrievalResult]] = retriever.retrieve_batch([query], return_scores=True)

Searching queries: 100%|██████████| 1/1 [00:00<00:00,  1.59it/s]


In [43]:
# Evaluate the retrieval results
all_recall = {}
for idx in range(len(all_results)):
    pred_pids = [result.doc.id for result in all_results[idx]]
    recall = get_recall_rates(ranked_pids=pred_pids, gold_pids=[pids[idx]], metric="all")
    for key, value in recall.items():
        if key not in all_recall:
            all_recall[key] = []
        all_recall[key].append(value)
# Average the recall rates
for key, value in all_recall.items():
    all_recall[key] = sum(value) / len(value)
print(all_recall)

{'@5': 0.0, '@10': 0.0, '@50': 0.0, '@100': 0.0}


In [44]:
pred_docs = [result.doc for result in all_results[0]]
pred_scores = [result.score for result in all_results[0]]
pred_pids = [doc.id for doc in pred_docs]
top_k = len(pred_pids)

# Find the gold pids
gold_founds = []
gold_found_indices = []
for gold_pid in gold_pids:
    gold_founds.append(gold_pid in pred_pids)
    if gold_pid in pred_pids:
        gold_found_indices.append(pred_pids.index(gold_pid))
    else:
        gold_found_indices.append(-1)

print(f"Is gold in Top-{top_k} docs?: {gold_founds}, at rank {gold_found_indices}")
show_passages(query=query, docs=pred_docs, scores=pred_scores, gold_pids=gold_pids, gold_ranks=gold_found_indices, collection=collection)

Is gold in Top-100 docs?: [True, True], at rank [0, 2]
Query:  What was the father of Kasper Schmeichel voted to be by the IFFHS in 1992?
Gold document 0: (PID: 590826) (score: 17.90625)
Kasper Peter Schmeichel (] ; born 5 November 1986) is a Danish professional footballer who plays as a goalkeeper for Premier League club Leicester City and the Denmark national team. He is the son of former Manchester United and Danish international goalkeeper Peter Schmeichel.

Gold document 1: (PID: 165560) (score: 17.15625)
Peter Boleslaw Schmeichel MBE (] ; born 18 November 1963) is a Danish former professional footballer who played as a goalkeeper, and was voted the IFFHS World's Best Goalkeeper in 1992 and 1993. He is best remembered for his most successful years at English club Manchester United, whom he captained to the 1999 UEFA Champions League to complete the Treble, and for winning UEFA Euro 1992 with Denmark.

Document 1: Kasper Schmeichel (PID: 590826)  (score: 17.90625)
Kasper Schmeichel

In [38]:
examine(retriever=retriever, query=query, pid=gold_pids[1], collection=collection, is_phrase=IS_PHRASE)

100%|██████████| 1/1 [00:00<00:00, 46.75it/s]


Query: Which performance act has a higher instrument to person ratio, Badly Drawn Boy or Wolf Alice?
Document (score: 13.642168045043945):
Wolf Alice | Wolf Alice are a four-piece alternative rock band from North London, formed initially as a two-person band in 2010. Its members since 2012 are Ellie Rowsell (vocals, guitar), Joff Oddie (guitars, vocals), Theo Ellis (bass), and Joel Amey (drums, vocals).

Query token 0: [CLS] (max score: 1.032275915145874)
Query token 1: [Q] (max score: 0.922183096408844)
Query token 2: which (max score: 0.7822329998016357)
Query token 3: performance act (max score: 0.4437413513660431)
Query token 4: has (max score: 0.5404560565948486)
Query token 5: a (max score: 0.6207960247993469)
Query token 6: higher (max score: 0.448029488325119)
Query token 7: instrument (max score: 0.4762474596500397)
Query token 8: to (max score: 0.5381997227668762)
Query token 9: person ratio (max score: 0.6252461671829224)
Query token 10: , (max score: 0.7028146386146545)
Que

In [40]:
examine(retriever=retriever, query=query, pid=309214, collection=collection, is_phrase=IS_PHRASE)

100%|██████████| 1/1 [00:00<00:00, 63.64it/s]


Query: Which performance act has a higher instrument to person ratio, Badly Drawn Boy or Wolf Alice?
Document (score: 13.676666259765625):
Wolf tone | "A wolf tone, or simply a ""wolf"", is produced when a played note matches the natural resonating frequency of the body of a musical instrument, producing a sustaining sympathetic artificial overtone that amplifies and expands the frequencies of the original note, frequently accompanied by an oscillating beating (due to the uneven frequencies between the natural note and artificial overtone) which may be likened to the howling of the animal. A similar phenomenon is the beating produced by a wolf interval, which is usually the interval between E and G# of the various non-circulating temperaments."

Query token 0: [CLS] (max score: 0.9565608501434326)
Query token 1: [Q] (max score: 0.9853193759918213)
Query token 2: which (max score: 0.6994242668151855)
Query token 3: performance act (max score: 0.6347508430480957)
Query token 4: has (max 